> **Companion notebook** — generated from `modules/04-likelihood.md` (the canonical, harness-verified text; regenerate with `python tools/make_notebooks.py`). Run cells top-to-bottom from the `curriculum/` directory so `figures/...` paths resolve. Cells marked *illustration only* are intentionally not executable.

# 04. Likelihood: the Data's Voice  [SIGNATURE S3]

> **Spine.** Identical data give an identical likelihood and therefore an identical posterior — even when they give *different* p-values; the evidence is the likelihood, not the intention behind the sampling.
> **Which line?** Line 2 (inference = conditioning). The likelihood $p(y\mid\theta)$ is the *entire channel* through which data reach the posterior — everything the data say about $\theta$ passes through this one function.
> **Promise.** After this module you can read a dataset's evidence off its likelihood, compress it losslessly to sufficient statistics, quantify its sharpness with Fisher information, and explain to a skeptic why two labs with the same counts can disagree on "significance" yet share a posterior.
> **Prereqs.** Modules 00 (four lines), 02 (conditioning is inference), 03 (generative stories, the information section).
> **Runtime.** ~4 s.
> **Sources.** C-B §6.2 (sufficiency, ancillarity), §6.3 (likelihood principle), §3.4 (exponential families); booklet ch. 2 §2.6; Berger–Wolpert *The Likelihood Principle* by concept; the 9-of-12 example after Lindley–Phillips by concept.

Recall the four lines (module 00): **a model is a joint $p(\text{unknowns},\text{knowns})$; inference is conditioning; prediction is marginalization; a decision minimizes posterior expected loss.** Conditioning reads $p(\theta\mid y)\propto p(y\mid\theta)\,p(\theta)$. The prior is yours; the *data's* only contribution is the factor $p(y\mid\theta)$, read as a function of $\theta$. This module is about that factor: what it is, how far it can be compressed, how sharply it speaks, and what it does — and does not — depend on.

In [ ]:
# --- setup ---
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats

SLUG = "04-likelihood"                   # this module's figure dir
FIG = Path("figures") / SLUG
FIG.mkdir(parents=True, exist_ok=True)
SEED = 0
rng = np.random.default_rng(SEED)

plt.rcParams.update({
    "figure.figsize": (7, 4), "figure.dpi": 110, "savefig.dpi": 150,
    "savefig.bbox": "tight", "axes.grid": True, "grid.alpha": 0.3,
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 11,
})

def save(fig, name):
    out = FIG / f"{name}.png"
    fig.savefig(out)
    plt.close(fig)
    print(f"[fig] {out}")

## 04.1 The likelihood function: the data's fingerprint on hypothesis space

**Definition (C-B 6.3.1).** Given the joint pmf/pdf $f(y\mid\theta)$ and an *observed* $y$, the function of $\theta$
$$L(\theta\mid y) = f(y\mid\theta)$$
is the **likelihood function**. Same formula as the sampling density, opposite reading: $f(y\mid\theta)$ varies $y$ with $\theta$ fixed and integrates to 1; $L(\theta\mid y)$ varies $\theta$ with $y$ fixed and integrates to nothing in particular. It is not a probability distribution over $\theta$ — it becomes one only after you multiply by a prior and normalize.

Its *shape* over $\theta$ is the data's fingerprint on hypothesis space: tall where the data are consistent with $\theta$, near-zero where they are not. For $n$ iid Bernoulli draws with $s=\sum_i y_i$ successes, $L(\theta) = \theta^{s}(1-\theta)^{n-s}$. Watch what accumulating data does to it.

In [ ]:
# The likelihood sharpens as data accumulate: same coin, growing n.
th_true = 0.6
y = rng.binomial(1, th_true, size=100)         # one long stream; take prefixes
grid = np.linspace(1e-3, 1 - 1e-3, 400)

fig, ax = plt.subplots()
for n in (5, 20, 100):
    yn = y[:n]; s = yn.sum()
    loglik = s*np.log(grid) + (n - s)*np.log(1 - grid)
    L = np.exp(loglik - loglik.max())          # normalize peak to 1 for display
    ax.plot(grid, L, label=f"n={n},  ∑x={s},  θ̂={s/n:.2f}")
    print(f"n={n:>3}:  ∑x={s:>3},  MLE θ̂={s/n:.3f}")
ax.axvline(th_true, ls="--", color="k", lw=1, label="truth θ=0.6")
ax.set_xlabel("θ"); ax.set_ylabel("normalized likelihood  L(θ)/max")
ax.set_title("The likelihood is the data's fingerprint — it sharpens with n")
ax.legend()
save(fig, "likelihood-growing")

![Normalized Bernoulli likelihood for n=5, 20, 100: broad at small n, sharply peaked at large n, peak location wobbling near 0.5–0.6.](figures/04-likelihood/likelihood-growing.png)

At $n=5$ the peak sits at $\hat\theta=$ `0.600` and the curve is broad — a wide range of $\theta$ is roughly as plausible as any other. At $n=100$ the MLE is `0.510` and the curve is a spike. Two things move at different rates: the peak *location* wobbles (`0.600`, `0.500`, `0.510`) because the MLE is noisy at small $n$, while the peak *width* shrinks like $1/\sqrt n$. That width — the sharpness of the data's voice — is exactly what Fisher information (§04.4) measures.

## 04.2 Sufficiency: when a summary is the whole message

The three prefixes above never used the *order* of the flips, only the count $s=\sum_i y_i$. That is not a shortcut; it is a theorem.

**Factorization Theorem (C-B 6.2.6).** $T(y)$ is **sufficient** for $\theta$ iff the joint factors as
$$f(y\mid\theta) = g\big(T(y)\mid\theta\big)\,h(y),$$
where $h$ does not involve $\theta$. Since the posterior is $p(\theta\mid y)\propto f(y\mid\theta)\,p(\theta) \propto g(T(y)\mid\theta)\,p(\theta)$, the $\theta$-free factor $h(y)$ cancels: **the posterior depends on the data only through $T(y)$.** A sufficient statistic captures everything the sample says about $\theta$ (under the assumed model), and discards the rest.

For Bernoulli, $f(y\mid\theta)=\theta^{\sum y_i}(1-\theta)^{n-\sum y_i}$ already has the factored form with $T=\sum y_i$ and $h\equiv1$ (C-B Example 6.2.3). So *conditional on $\sum y_i = s$*, every arrangement of the successes is equally likely, whatever $\theta$ is. Let us watch the order become irrelevant.

In [ ]:
# Shuffle test: condition on ∑x = 3 among 8 Bernoulli flips.
# If ∑x is sufficient, the conditional distribution over the C(8,3) arrangements
# is uniform AND identical for different θ. Simulate at θ=0.3 and θ=0.6.
from math import comb
n_bits, s_fix = 8, 3
n_pat = comb(n_bits, s_fix)
uniform = 1 / n_pat
powers = 1 << np.arange(n_bits)                # bit-pattern -> integer code
print(f"arrangements C(8,3) = {n_pat},  uniform prob = {uniform:.4f}")
for theta in (0.3, 0.6):
    X = rng.binomial(1, theta, size=(400_000, n_bits))
    keep = X[X.sum(1) == s_fix]                # condition on the sufficient stat
    _, counts = np.unique(keep @ powers, return_counts=True)
    freq = counts / counts.sum()
    print(f"θ={theta}:  max|conditional freq − uniform| = {np.abs(freq - uniform).max():.4f}")

Every arrangement lands within `0.0009` (at $\theta=0.3$) and `0.0014` (at $\theta=0.6$) of the uniform probability `0.0179` — flat, and the same flat for both $\theta$. The order of the flips is Monte-Carlo noise; the count is the message. This is line 2 in miniature: conditioning on $y$ and conditioning on $T(y)$ give the same posterior.

**The sufficiency collision.** Push this to its sharpest form. For $N(\mu,\sigma^2)$ with both unknown, the sufficient statistic is $\big(\sum x_i,\ \sum x_i^2\big)$ (C-B Example 6.2.9). Below are two datasets, $A$ and $B$, that differ in three of their six entries. *Predict before running:* will their posteriors for $(\mu,\sigma^2)$ differ — and if so, in which parameter, and by roughly how much? The natural read — "different data must move the posterior at least a little" — is the one on trial. The conjugate Normal–Inverse-Gamma update (previewed here, derived in module 05) touches the data only through $n$ and those two sums:

In [ ]:
# NIG posterior parameters depend on data ONLY through (n, Σx, Σx²).
def nig_post(data, mu0=0.0, k0=1.0, a0=1.0, b0=1.0):
    n = len(data); xbar = data.mean(); S = ((data - xbar)**2).sum()  # S = Σx² − nx̄²
    kappa_n = k0 + n
    mu_n    = (k0*mu0 + n*xbar) / kappa_n
    alpha_n = a0 + n/2
    beta_n  = b0 + 0.5*S + (k0*n*(xbar - mu0)**2) / (2*kappa_n)
    return kappa_n, mu_n, alpha_n, beta_n

A = np.array([1., 5.,  6., 10., 11., 12.])     # two visibly different datasets
B = np.array([2., 3.,  7., 10., 11., 12.])
print(f"Σx: {A.sum():.0f} vs {B.sum():.0f}   Σx²: {(A**2).sum():.0f} vs {(B**2).sum():.0f}")
pa, pb = nig_post(A), nig_post(B)
print(f"posterior A: κ_n={pa[0]:.1f}  μ_n={pa[1]:.4f}  α_n={pa[2]:.1f}  β_n={pa[3]:.4f}")
print(f"posterior B: κ_n={pb[0]:.1f}  μ_n={pb[1]:.4f}  α_n={pb[2]:.1f}  β_n={pb[3]:.4f}")
print(f"identical posterior parameters: {pa == pb}")

Datasets $A$ and $B$ differ in their first three entries but share $\sum x = $ `45` and $\sum x^2 = $ `427`. Their posteriors are the same tuple to the last digit — $\mu_n=$ `6.4286`, $\beta_n=$ `69.8571` — because a Normal model literally cannot see past the two sums. **Sufficiency is lossless compression, but only relative to the model** (a Pitfall below: change the model and the collision unlocks).

## 04.3 Exponential families, compressed

You met these stories in module 03; here is the algebraic skeleton they share (C-B §3.4). A family is an **exponential family** if
$$f(x\mid\theta) = h(x)\,c(\theta)\,\exp\!\Big(\textstyle\sum_{i=1}^{k} w_i(\theta)\,t_i(x)\Big),$$
and in **natural (canonical) form**, with natural parameter $\eta$, log-partition $A(\eta)$, and sufficient statistic $T$,
$$p(x\mid\eta) = h(x)\,\exp\!\big(\eta^\top T(x) - A(\eta)\big),\qquad A(\eta)=\log\!\int h(x)e^{\eta^\top T(x)}dx.$$
By the Factorization Theorem the per-observation $T$ sums across iid data, so $\sum_i T(x_i)$ is sufficient at every $n$ (C-B Theorem 6.2.10) — the reason closed-form posteriors exist at all (cashed out in module 05).

| family | natural parameter $\eta$ | sufficient statistic $T(x)$ |
|---|---|---|
| Bernoulli($\theta$) | $\log\frac{\theta}{1-\theta}$ (logit) | $x$ |
| Poisson($\lambda$) | $\log\lambda$ | $x$ |
| Normal($\mu,\sigma^2$) | $\big(\mu/\sigma^2,\ -1/(2\sigma^2)\big)$ | $(x,\ x^2)$ |
| Gamma($\alpha,\beta$), $\beta$=rate | $\big(\alpha-1,\ -\beta\big)$ | $(\log x,\ x)$ |

The log-partition $A$ is a moment generator: differentiating it gives the cumulants of $T$,
$$A'(\eta) = \mathbb{E}[T(X)],\qquad A''(\eta) = \mathrm{Var}[T(X)]$$
(this is C-B Theorem 3.4.2 in canonical coordinates). For Poisson, $\eta=\log\lambda$, $T=x$, and $A(\eta)=e^{\eta}=\lambda$, so both derivatives equal $\lambda$ — Poisson's signature mean-equals-variance falls out of one function.

In [ ]:
# ONE numerical check: Poisson cumulants A'(η)=E[T], A''(η)=Var[T].
lam = 3.7
xp = rng.poisson(lam, size=200_000)
eta = np.log(lam)
Aprime, Adprime = np.exp(eta), np.exp(eta)     # A'(η)=A''(η)=e^η=λ
print(f"A'(η)  = {Aprime:.4f}   sample mean     = {xp.mean():.4f}")
print(f"A''(η) = {Adprime:.4f}   sample variance = {xp.var():.4f}")

The theory says $A'=A''=$ `3.7000`; the data return a sample mean of `3.7001` and a sample variance of `3.6944`. Mean and variance agree because for Poisson they are *the same cumulant*.

**Pitman–Koopman–Darmois (Theorem; Pitman 1936, Koopman 1936, Darmois 1935 — see Lehmann–Casella, *Theory of Point Estimation*, by concept).** Among families whose support does not depend on $\theta$ (plus smoothness conditions), the exponential families are *exactly* those admitting a sufficient statistic of fixed dimension for every $n$. Two foils sharpen it. The uniform $U(0,\theta)$ *does* have a one-number sufficient statistic, $\max_i x_i$ — but only by breaking the premise: its support $(0,\theta)$ carries $\theta$, so the information leaks out through the boundary rather than through a sum. The Cauchy location family keeps fixed support and pays the price the theorem promises: its minimal sufficient statistic is the *entire ordered sample*, growing with $n$ (Exercise 04.3). Fixed-dimensional summaries are an exponential-family privilege.

## 04.4 Fisher information: the curvature of the data's voice

How loudly does the data's voice speak at a given $\theta$? Define the **score** as the slope of the log-likelihood,
$$U(\theta) = \frac{\partial}{\partial\theta}\log f(X\mid\theta).$$
Under mild regularity (differentiate $\int f\,dx=1$ under the integral), the score has mean zero and its variance is the **Fisher information**:
$$\mathbb{E}_\theta[U(\theta)] = 0,\qquad \mathcal{I}(\theta) = \mathrm{Var}_\theta[U(\theta)] = -\,\mathbb{E}_\theta\!\left[\frac{\partial^2}{\partial\theta^2}\log f(X\mid\theta)\right].$$
The two right-hand expressions are equal under the same regularity — variance of the slope, or expected negative curvature. For one Bernoulli($\theta$) draw, $\log f = x\log\theta + (1-x)\log(1-\theta)$, so $U(\theta)=\frac{x}{\theta}-\frac{1-x}{1-\theta}$ and a line of algebra gives $\mathcal{I}(\theta)=\frac{1}{\theta(1-\theta)}$. Verify by simulation at $\theta=0.3$:

In [ ]:
theta0 = 0.3
I0 = 1 / (theta0*(1 - theta0))                 # analytic Fisher information
xb = rng.binomial(1, theta0, size=2_000_000)
score = xb/theta0 - (1 - xb)/(1 - theta0)      # U(θ) for each draw
print(f"I(θ) = 1/(θ(1−θ)) = {I0:.4f}")
print(f"E[score]   (sim) = {score.mean():.4f}    (exact 0)")
print(f"Var[score] (sim) = {score.var():.4f}    (exact {I0:.4f})")

The simulated score averages `0.0007` (the exact zero) and has variance `4.7632`, matching $\mathcal{I}(0.3)=$ `4.7619`. Information is **additive**: for $n$ iid observations the scores are independent with mean zero, so their variances add and $\mathcal{I}_n(\theta)=n\,\mathcal{I}(\theta)$. Geometrically, information is curvature — how sharply the log-likelihood bends at its peak.

In [ ]:
# Curvature grows with n: log-likelihood peak sharpens, second-derivative = n·I.
fig, ax = plt.subplots()
for n, color in ((10, "C0"), (200, "C1")):
    yn = rng.binomial(1, theta0, size=n); s = yn.sum(); thhat = s/n
    ll = s*np.log(grid) + (n - s)*np.log(1 - grid)
    obs_info = n / (thhat*(1 - thhat))          # −ℓ''(θ̂) = n·I(θ̂)
    ax.plot(grid, ll - ll.max(), color=color,
            label=f"n={n}:  θ̂={thhat:.2f},  curvature n·I(θ̂)={obs_info:.0f}")
    print(f"n={n:>3}: θ̂={thhat:.3f}, observed information n·I(θ̂)={obs_info:.1f}")
ax.set_xlim(0, 0.8); ax.set_ylim(-6, 0.3)
ax.set_xlabel("θ"); ax.set_ylabel("log-likelihood (peak at 0)")
ax.set_title("Fisher information = curvature of the log-likelihood")
ax.legend()
save(fig, "fisher-curvature")

![Two Bernoulli log-likelihood curves: n=10 broad and gently curved, n=200 narrow and sharply curved, curvature scaling with n.](figures/04-likelihood/fisher-curvature.png)

The $n=10$ curve is gently rounded (observed information `41.7`); the $n=200$ curve is a sharp cusp (`952.4` $= 200\times\mathcal{I}(0.3)$). This curvature is the whole plot of the sequel: module 07 builds the **Jeffreys prior** $\propto\sqrt{\mathcal{I}(\theta)}$ (a construction rule that is invariant to reparameterization — module 07 makes this precise), and module 08 shows the MLE is asymptotically $\hat\theta\sim N\!\big(\theta,\ 1/(n\mathcal{I}(\theta))\big)$ — the posterior's width is $1/\sqrt{n\mathcal{I}}$, so more curvature means a tighter answer.

So far the likelihood has told us what the data say (§§04.1–04.2) and how sharply (§04.4). One question remains: what does it *not* depend on?

## 04.5 Two labs, one coin  [SIGNATURE S3]

**Setup.** Two labs test whether a coin favors heads ($H_0:\theta=0.5$). Both observe the *same result*: **9 heads and 3 tails in 12 flips.** They differ only in the plan.
- **Lab F (fixed sample size)** decided in advance to flip exactly 12 times, then count heads. Their count $X$ is Binomial$(12,\theta)$.
- **Lab S (sequential)** decided to flip *until the 3rd tail*, then stop. The 3rd tail happened to land on flip 12. Their count of heads $X$ before the 3rd tail is Negative-Binomial (3 failures, $\theta$).

**Predict.** Same coin, same 9-heads-in-12 result, same null. Before running anything, commit to **numbers**: write down your guess for each lab's one-sided p-value — or, at minimum, which side of 0.05 each lands on — and whether the two verdicts at $\alpha=0.05$ agree. The intuition nearly everyone brings is that this is one computation done twice: *identical data, identical arithmetic, identical p.* Write your two numbers down and hold onto them.

**Run.** Each lab computes a one-sided p-value: the null probability of a result *at least as extreme* (as many heads or more) as observed.

In [ ]:
# Same data (9 heads, 3 tails), two sampling plans, two p-values.
from math import comb
theta_null = 0.5                                           # H0 value of θ
p_binom = stats.binom(12, theta_null).sf(8)                # P(X ≥ 9), n fixed at 12
# CONVENTION TRAP: scipy's nbinom(n, p) counts FAILURES before the n-th SUCCESS
# with success prob p. Here the stopping event is a TAIL (prob 1−θ) and we count
# heads, so heads = scipy-"failures" and the correct call is nbinom(3, 1−θ).
# (At θ=0.5 the symmetric call nbinom(3, θ) happens to give the same answer.)
p_negbin = stats.nbinom(3, 1 - theta_null).sf(8)           # P(X ≥ 9 heads before 3rd tail)
# hand-computation cross-check (STYLE §6: print both routes)
p_binom_hand  = sum(comb(12, k) for k in range(9, 13)) / 2**12
p_negbin_hand = 1 - sum(comb(k + 2, 2) * 0.5**k * 0.5**3 for k in range(0, 9))
print(f"Lab F  binomial   one-sided p = {p_binom:.3f}   (hand {p_binom_hand:.3f})")
print(f"Lab S  neg-binom  one-sided p = {p_negbin:.3f}   (hand {p_negbin_hand:.3f})")
print(f"9-of-12 shared MLE θ̂ = 9/12 = {9/12:.2f}")

At the conventional 0.05 line, **Lab F reports $p=$ `0.073`: not significant.** **Lab S reports $p=$ `0.033`: significant.** If you wrote down one p-value for both labs — the near-universal guess — you have just been caught: the two designs do not even agree on which *side* of 0.05 the same 12 flips fall. Same coin, same flips — opposite verdicts.

**Reconcile.** Why do the p-values differ, and what does *not* differ? A p-value sums probability over outcomes that *did not happen* — the tail "9 or more heads." But the two designs have different sample spaces of might-have-beens: Lab F could have produced any of 0–12 heads in 12 flips; Lab S could have run for any number of flips. Different might-have-beens, different tails, different p-values. Now look at what the *observed* data actually contribute — the likelihood:
$$L_F(\theta) = \binom{12}{9}\theta^9(1-\theta)^3,\qquad L_S(\theta) = \binom{11}{9}\theta^9(1-\theta)^3.$$
The binomial coefficients differ, but they are constants in $\theta$. As functions of $\theta$ the two likelihoods are **proportional** — the same curve up to a scale factor.

In [ ]:
# The two designs give proportional likelihoods: ratio is constant in θ.
tg = np.array([0.2, 0.5, 0.8])
L_F = comb(12, 9) * tg**9 * (1 - tg)**3
L_S = comb(11, 9) * tg**9 * (1 - tg)**3
print(f"L_F / L_S at θ=0.2,0.5,0.8 = {np.round(L_F / L_S, 4)}   (constant {comb(12,9)/comb(11,9):.1f})")

# Identical likelihood ⇒ identical posterior. Beta(1,1) prior + (9 succ, 3 fail):
a_post, b_post = 1 + 9, 1 + 3
print(f"both labs' posterior = Beta({a_post},{b_post}),  mean = {a_post/(a_post + b_post):.4f}")

fig, ax = plt.subplots(1, 3, figsize=(12, 4.0))
k = np.arange(13)
ax[0].bar(k, stats.binom(12, 0.5).pmf(k), color="C7")
ax[0].bar(k[k >= 9], stats.binom(12, 0.5).pmf(k[k >= 9]), color="C3")
ax[0].set_title(f"Lab F sample space:  P(X≥9)={p_binom:.3f}"); ax[0].set_xlabel("# heads in 12")
kk = np.arange(25)
ax[1].bar(kk, stats.nbinom(3, 0.5).pmf(kk), color="C7")
ax[1].bar(kk[kk >= 9], stats.nbinom(3, 0.5).pmf(kk[kk >= 9]), color="C3")
ax[1].set_title(f"Lab S sample space:  P(X≥9)={p_negbin:.3f}"); ax[1].set_xlabel("# heads before 3rd tail")
Lg = grid**9 * (1 - grid)**3; Lg /= Lg.max()
post = stats.beta(a_post, b_post).pdf(grid)
ax[2].plot(grid, Lg, color="C0", lw=4, label="likelihood ∝ θ⁹(1−θ)³")
ax[2].plot(grid, post/post.max(), color="C1", lw=1.8, ls="--",
           label=f"posterior Beta({a_post},{b_post}) — coincides")
ax[2].axvline(0.75, ls=":", color="k", lw=1)
ax[2].set_title("Shared by BOTH labs"); ax[2].set_xlabel("θ"); ax[2].legend(fontsize=8)
fig.tight_layout(rect=(0, 0, 1, 0.92))
fig.suptitle("Same 9-of-12 data: the design splits the p-value, not the likelihood")
save(fig, "same-data-opposite-verdicts")

![Three panels: Lab F binomial sample space with tail p=0.073; Lab S negative-binomial sample space with tail p=0.033; and the single shared likelihood and Beta(10,4) posterior common to both.](figures/04-likelihood/same-data-opposite-verdicts.png)

The ratio $L_F/L_S$ is `4.0` at every $\theta$. Feed either likelihood into a flat Beta(1,1) prior and both labs land on the *same* posterior, **Beta(10,4)** with mean `0.7143`, peaked at the shared MLE `0.75`. In the figure's right panel the dashed posterior rides exactly on the solid likelihood — with a flat prior the posterior *is* the likelihood renormalized. Conditioning — line 2 — sees only the likelihood, and the likelihood cannot tell the designs apart. The two labs differ in their *intention* (when to stop), not in their *evidence*. **The posterior is deaf to intentions; the p-value is not.**

### The likelihood principle, stated honestly

**Likelihood Principle (C-B §6.3).** If $L(\theta\mid x) = C(x,y)\,L(\theta\mid y)$ for all $\theta$ — the two likelihoods proportional — then $x$ and $y$ carry the *same evidence* about $\theta$, and any post-data inference should treat them identically. Birnbaum's theorem (C-B 6.3.6) derives this from two premises that are individually widely accepted — the **sufficiency principle** (§04.2) and the **conditionality principle** (base inference on the experiment actually performed) — though the formal versions used in the proof, and the proof itself, have their critics. Bayesian conditioning obeys the LP automatically, because the posterior only ever multiplies the likelihood.

This is where a course can turn tribal, and this one refuses to (per the brief). **The frequentist p-value is not a mistake.** It is the exact answer to a *different question*: "if $H_0$ held and I reran *this design* forever, how often would I see evidence this strong?" That question legitimately depends on the design's sample space — on outcomes that did not occur — because it is about the long-run behavior of a *procedure*, not about *these* data. The posterior answers "given exactly what I saw, what should I believe about $\theta$?" and depends only on the likelihood. Two coherent answers to two different questions.

> **Scope box — where stopping rules *do* matter.** The LP governs **post-data inference** (what these data say about $\theta$). It does **not** govern **design** (which data to collect, when to stop). At the planning stage you have no likelihood yet — only a distribution over data you might get — so stopping rules, power, and error rates are exactly the right currency; that is module 23's subject. The clean statement: *the reason you stopped cannot change what the data you got mean, but it absolutely governs what data you are likely to get.* Optional stopping does not bias a Bayesian's posterior (the §04.5 proportional-likelihood logic, proved in module 23), yet it inflates a frequentist Type-I rate — roughly 4× at ~10 interim looks, and without bound under continuous peeking (module 23 simulates both) — because those are guarantees about the procedure, not the datum.

## 04.6 Two seeds: censoring and ancillarity

**Censored data (a seed for module 15).** The likelihood is *the probability the model assigns to exactly what you observed* — no more, no less. Usually that is a density $f(y\mid\theta)$. But suppose a unit is still alive when the study ends at time $c$: all you know is $Y>c$. The model's probability of *that* observation is the survival function $S(c)=P(Y>c)=1-F(c)$, so a censored point contributes $S(c\mid\theta)$ to the likelihood instead of a density. Everything else — multiply the per-observation contributions, maximize or condition — is unchanged.

In [ ]:
# Exponential lifetimes: 5 exact deaths + 1 unit still alive at c=3.
# L(λ) = Π λe^{−λ y_i}  ·  S(c),  with S(c)=e^{−λc}.  ⇒ MLE = #deaths / total time-at-risk.
from scipy.optimize import minimize_scalar
obs, c = np.array([0.8, 1.2, 0.5, 2.1, 1.7]), 3.0
neg_ll = lambda lam: -(np.sum(np.log(lam) - lam*obs) - lam*c)      # note the −λc censoring term
lam_cens  = minimize_scalar(neg_ll, bounds=(1e-3, 10), method="bounded").x
lam_naive = (len(obs) + 1) / (obs.sum() + c)          # WRONG: treats the censored unit as a death
print(f"censored MLE (correct) = {lam_cens:.4f}   [= 5 deaths / {obs.sum()+c:.1f} time-at-risk]")
print(f"naive MLE (censor→death) = {lam_naive:.4f}   (over-estimates the hazard)")

Handling the censoring correctly gives rate `0.5376`; pretending the survivor died at $c$ inflates it to `0.6452`, because you credited a death that never happened. Same likelihood logic as the 9-of-12 story — write down the probability of *precisely* what you saw — now with partial observation. Module 15 builds survival regression on exactly this.

**Ancillarity (Basu's theorem; returns in module 06).** Ancillarity is sufficiency's complement: a sufficient statistic keeps *everything* the data say about $\theta$; an **ancillary** statistic is a piece of the data that says *nothing* about $\theta$ by itself — formally, its distribution does not depend on $\theta$ (C-B 6.2.16). For $N(\theta,1)$, the sample mean $\bar x$ is (complete) sufficient, while the residuals $x_i-\bar x$ are ancillary: shifting $\theta$ slides every $x_i$ equally and leaves the residuals untouched. **Basu's theorem (C-B 6.2.24)** says a complete sufficient statistic is independent of *every* ancillary (C-B states it with "minimal" added to the hypothesis; that assumption isn't needed) — so here $\bar x \perp (x_i-\bar x)$, exactly.

In [ ]:
Xs = rng.normal(5.0, 1.0, size=(200_000, 6))          # θ=5, n=6
xbar = Xs.mean(1)
corr_resid = np.corrcoef(xbar, Xs[:, 0] - xbar)[0, 1]  # x̄ vs a residual
corr_svar  = np.corrcoef(xbar, Xs.var(1, ddof=1))[0, 1]# x̄ vs sample variance (a fn of residuals)
print(f"corr(x̄, x₁−x̄) = {corr_resid:.4f}")
print(f"corr(x̄, s²)   = {corr_svar:.4f}")

Both correlations are simulation-zero — `-0.0026` and `-0.0020` — the fingerprint of exact independence. This is why "estimate $\theta$ with $\bar x$" and "check the model with the spread" are separable tasks for a Normal: the location estimate and the shape diagnostic are independent. Module 06 uses ancillary statistics to decide *what to condition on* when a procedure's guarantee depends on it.

## Bridge — likelihood → loss: the ML dictionary

Every loss function you have minimized in a machine-learning course is a negative log-likelihood in disguise, and **maximum likelihood is empirical risk minimization under log loss**: $\hat\theta = \arg\min_\theta \frac1n\sum_i \big[-\log p(y_i\mid\theta)\big]$. Choosing a loss *is* choosing a noise model.

| ML loss | $=$ NLL of | minimizer is |
|---|---|---|
| binary cross-entropy | Bernoulli / categorical | class frequency |
| mean squared error (MSE) | Gaussian (fixed $\sigma$) | sample mean |
| mean absolute error (MAE) | Laplace | sample median |

In [ ]:
# Minimize each loss numerically; recover the corresponding MLE statistic.
yb = rng.binomial(1, 0.35, size=5000)
bce = lambda t: -(yb*np.log(t) + (1 - yb)*np.log(1 - t)).mean()
t_bce = minimize_scalar(bce, bounds=(1e-4, 1 - 1e-4), method="bounded").x
print(f"argmin cross-entropy = {t_bce:.4f}   sample mean (Bernoulli MLE) = {yb.mean():.4f}")

yg = rng.normal(2.0, 1.5, size=5000)
m_mse = minimize_scalar(lambda m: ((yg - m)**2).mean(),    bounds=(-5, 10), method="bounded").x
m_mae = minimize_scalar(lambda m: np.abs(yg - m).mean(),   bounds=(-5, 10), method="bounded").x
print(f"argmin MSE = {m_mse:.4f}   sample mean   (Gaussian MLE) = {yg.mean():.4f}")
print(f"argmin MAE = {m_mae:.4f}   sample median (Laplace  MLE) = {np.median(yg):.4f}")

Minimizing cross-entropy returns `0.3416`, exactly the sample frequency and the Bernoulli MLE. MSE returns `2.0267`, the sample mean and Gaussian MLE. MAE returns `2.0179`, matching the sample median `2.0178` to optimizer tolerance (the absolute-loss surface is non-smooth, so the minimizer lands one step off the exact median). Cross-entropy = expected surprise, the information-theoretic quantity from module 03; here it wears its statistical costume. GLMs formalize the general recipe — likelihood + link — in module 15.

## Pitfalls

- **Reading $L(\theta)$ as a density over $\theta$.** The likelihood is *not* a probability distribution: it does not integrate to 1 in $\theta$, and its height is meaningful only up to a constant. Only $L(\theta)\,p(\theta)$, normalized, is a distribution. "The likelihood at $\hat\theta$ is 0.28" says nothing on its own.
- **Forgetting sufficiency is model-relative.** $\sum x_i$ is sufficient *for the Bernoulli model*; $(\sum x_i,\sum x_i^2)$ *for the Normal*. If the model is wrong, the sufficient statistic can discard exactly the evidence of the wrongness — the sufficiency collision unlocks the moment you doubt normality (the differing minima — 1 vs 2 — and raw values of $A$ and $B$ suddenly matter for model checking, module 17). Losslessness is a promise conditional on the model.
- **Confusing Fisher information with observed information.** $\mathcal{I}(\theta)=\mathbb{E}[-\ell'']$ averages over data; the **observed** information $-\ell''(\hat\theta)$ does not. They agree asymptotically, but the thing you actually compute from one dataset is the observed version — and conditioning on ancillaries (module 06) can make it the *right* thing to use.
- **Calling the stopping-rule dependence of the p-value a paradox.** It is the correct behavior of a pre-data error rate, which must integrate over the design's sample space. The error is expecting that same number to also be a post-data measure of evidence — two different jobs (§04.5).
- **Plugging censored observations in as if exact.** A survivor recorded as a death at $c$ inflates the hazard (`0.6452` vs `0.5376` above). The fix is not a heuristic correction; it is writing the correct likelihood contribution $S(c)$.

## Exercises

**Exercise 04.1 — Does the proportion tell the whole story?**
*Setup:* Lab A observes 9 successes in 12 trials; Lab B observes 90 in 120. Both use a flat Beta(1,1) prior.
*Predict:* Same success proportion (0.75). Is Lab B's posterior for $\theta$ meaningfully sharper than Lab A's, or about the same?
*Reason:* The intuition "same proportion ⇒ same conclusion" ignores that evidence accumulates with $n$ — the Fisher-information lesson of §04.4.
*Run:*

In [ ]:
for s, n in [(9, 12), (90, 120)]:
    a, b = 1 + s, 1 + (n - s)
    print(f"{s}/{n}: Beta({a},{b})  sd={stats.beta(a, b).std():.4f}  mean={a/(a+b):.4f}")

<details><summary>Reconcile</summary>

Lab A gets Beta(10,4), posterior sd `0.1166`; Lab B gets Beta(91,31), sd `0.0393` — about three times ($\approx\sqrt{10}$) tighter. Information scales with $n$: ten times the data gives a $\sqrt{10}$-times *narrower* posterior sd, because posterior width $\sim 1/\sqrt{n\mathcal{I}}$ (the measured ratio, 2.97 rather than 3.16, is the flat prior's pseudo-counts mattering less at large $n$). The proportion sets the *location* of the likelihood; $n$ sets its *width*. (Lab B's mean, `0.7459`, also drifts slightly from 0.75 as the near-flat prior's one pseudo-success matters less.) A ratio quoted without a sample size is half a fact.
</details>

**Exercise 04.2 — Why train with cross-entropy instead of accuracy?**  *(ML bridge, surprising)*
*Setup:* Fit a single probability $\theta$ to 400 Bernoulli(0.3) labels, once by minimizing cross-entropy (log loss), once by minimizing 0-1 loss (error rate of the constant predictor "$\hat y = 1$ iff $\theta>0.5$").
*Predict:* Which objective recovers the true rate 0.3, and which is happy with a whole *range* of $\theta$?
*Reason:* "A classifier is judged by accuracy, so optimize accuracy" — the reflex that log loss exists only for differentiability.
*Run:*

In [ ]:
rng2 = np.random.default_rng(7)
y = rng2.binomial(1, 0.3, 400)
ce = lambda t: -(y*np.log(t) + (1 - y)*np.log(1 - t)).mean()
t_ce = minimize_scalar(ce, bounds=(1e-3, 1 - 1e-3), method="bounded").x
zo = lambda t: (int(t > 0.5) != y).mean()
print(f"min cross-entropy at θ={t_ce:.4f}   (sample freq {y.mean():.4f})")
print(f"0-1 loss at θ=0.1, 0.3, 0.49: {zo(0.1):.4f}, {zo(0.3):.4f}, {zo(0.49):.4f}")

<details><summary>Reconcile</summary>

Cross-entropy is minimized at `0.3175`, the exact sample frequency (the Bernoulli MLE) — it recovers a calibrated *probability*. The 0-1 loss is `0.3175` at $\theta=0.1$, at $0.3$, and at $0.49$ — flat for every $\theta<0.5$, because the constant predictor's error rate depends only on *which side of 0.5* $\theta$ sits, not on its value. Accuracy cannot identify a probability; log loss (a strictly proper scoring rule = a genuine likelihood) can. This is precisely why classifiers are *trained* on cross-entropy and only *reported* on accuracy — and why a model can be accurate yet badly miscalibrated (module 15).
</details>

**Exercise 04.3 — Is $\sum x_i$ always sufficient?**  *(module 03 callback, surprising)*
*Setup:* Draw five samples of 1000 iid Cauchy variates centered at 5. The Cauchy location family has *fixed* support and is *not* an exponential family.
*Predict:* The sample mean of 1000 draws — will it land near the center 5, the way a Normal's would?
*Reason:* "Average lots of data and you get the center" — the CLT habit, applied where it does not hold.
*Run:*

In [ ]:
rng3 = np.random.default_rng(3)
xc = rng3.standard_cauchy((5, 1000)) + 5.0
print("means  :", ", ".join(f"{m:.2f}" for m in xc.mean(1)))
print("medians:", ", ".join(f"{m:.2f}" for m in np.median(xc, 1)))

<details><summary>Reconcile</summary>

The five sample means come out like `13.23`, `5.08`, `5.37`, `4.42`, `5.11` — one is off by more than 8, and averaging *does not* rein it in ($\bar X$ of Cauchy is again Cauchy, module 03). The medians cluster tightly around 5 (`5.09`, `5.00`, `4.93`, …). Keep the two facts separate: $\sum x_i$ is not sufficient for the Cauchy center *at all* — the minimal sufficient statistic is the whole ordered sample (Pitman–Koopman) — and this demo shows the *cost* of compressing to it anyway: the mean of 1000 draws is no better than a single draw. Fixed-dimensional sufficiency ($\bar x$ says it all) is an *exponential-family* privilege; outside that class, robust summaries like the median do the work.
</details>

**Exercise 04.4 — When does the crude normal approximation stop being crude?**  *(seeds module 08)*
*Setup:* From $n=100$ trials with 30 successes, form a 95% interval for $\theta$ two ways: the Fisher-information normal approximation $\hat\theta \pm 1.96/\sqrt{n\mathcal{I}(\hat\theta)}$, and the exact Beta(31,71) credible interval (flat prior).
*Predict:* The normal approximation ignores skew and boundaries — will it be far from the exact interval, or close?
*Reason:* "Asymptotic $=$ only-in-the-limit, so at $n=100$ expect a visible gap."
*Run:*

In [ ]:
s, n = 30, 100
that = s/n; se = 1/np.sqrt(n/(that*(1 - that)))
print(f"Fisher-normal 95%: [{that-1.96*se:.3f}, {that+1.96*se:.3f}]")
print(f"exact Beta  95%: [{stats.beta(1+s,1+n-s).ppf(0.025):.3f}, {stats.beta(1+s,1+n-s).ppf(0.975):.3f}]")

<details><summary>Reconcile</summary>

Fisher-normal gives `[0.210, 0.390]`; exact Beta gives `[0.219, 0.396]` — the endpoints agree to about one part in a hundred already at $n=100$. This is the **Bernstein–von Mises** phenomenon previewed: as information accumulates the posterior becomes Gaussian with covariance $1/(n\mathcal{I})$, so the curvature you measured in §04.4 *is* the posterior width, and the frequentist and Bayesian intervals fuse. Module 08 makes this a theorem and shows exactly where it breaks (boundaries, unidentified parameters, infinitely many nuisance parameters).
</details>

## Takeaways

- The **likelihood** $L(\theta)=f(y\mid\theta)$, read as a function of $\theta$, is the *entire* channel from data to posterior; conditioning multiplies it by the prior and normalizes. It is not a density over $\theta$.
- A **sufficient statistic** factorizes the likelihood and so carries all of the data's evidence about $\theta$ — losslessly, *relative to the model*. Two datasets with the same sufficient statistic have byte-identical posteriors.
- Among families with **fixed support** (plus smoothness), **exponential families** are exactly the ones with fixed-dimensional sufficient statistics — $U(0,\theta)$ escapes only by moving $\theta$ into the support; the log-partition's derivatives are the cumulants of that statistic (Poisson: mean $=$ variance $= \lambda$).
- **Fisher information** $\mathcal{I}(\theta)=\mathrm{Var}[\text{score}]=\mathbb{E}[-\ell'']$ is the curvature of the log-likelihood; it is additive ($n\mathcal{I}$) and sets the posterior width $1/\sqrt{n\mathcal{I}}$ — the seed of Jeffreys priors (07) and asymptotic normality (08).
- **S3:** the same 9-of-12 data give p-values `0.073` (fixed-$n$) and `0.033` (stop-at-3rd-tail) but one shared likelihood $\propto\theta^9(1-\theta)^3$ and one shared posterior Beta(10,4). The design splits the p-value; it cannot split the evidence.
- The **likelihood principle** governs post-data inference (obey it) but *not* design (stopping rules and error rates are the right currency there — module 23). The p-value is a correct answer to a different question, not a mistake.
- Every ML loss is a **negative log-likelihood**: cross-entropy = Bernoulli, MSE = Gaussian, MAE = Laplace; MLE = ERM under log loss. Censored observations contribute $S(c)$; ancillary statistics ($\bar x \perp$ residuals) carry no information about $\theta$ alone.